## NB For Testing / Recording Bitboard Implementation

Actual implementation is made in Cython file tablut.pyx

Board representation is 2 quad words in struct 'Bitboard'. First 63 bits (0-62) of Bitboard.low represents the 'bottom 7' ranks, first 18 bits (0-17) of Bitboard.high represents the 'top 2' ranks. I used this representation as it was the approach taken in the most sophisticated 9x9 open source board game implementation I readily found - Shogi program [Apery](https://github.com/HiraokaTakuya/apery_rust/blob/master/src/bitboard.rs). I believe this also affords us the elegant solution of merging the high and low boards with a simple bitwise OR function when creating a key for move generation.

### Checklist
__Groundwork__
- Decide on architecture and define it in a struct
- Write bit manipulation macros that work across both quad words
- Write basic board setting / clearing / printing functions

__Move Generation__
- Calculate moves 'on the fly'
- Pre-calculate blocker masks for all squares
- Generate keys from blocker masked board states
- Use Carry-Rippler trick to generate all possible blocker states for each square
- Use brute force trial and error to find magic numbers
- create a move table for index / move pairs

__Move Making__
- Create multiple Bitboards to track pieces
- Mask occupied Bitboard with move mask for legal moves

__Attacks__
- TBD (mask two over allies, map this to attacked squares, AND this with opponents for attacks?)

### Import our tablut Cython Module:

In [1]:
import tablut

### Creating our Board:

Board representation is 2 quad words in struct 'Bitboard'. First 63 bits (0-62) of Bitboard.low represents the 'bottom 7' ranks, first 18 bits (0-17) of Bitboard.high represents the 'top 2' ranks. I messed around with a few other approaches (1x uint64 + 1x uint32; 3x uint32), but decided on the 2x uint64 approach after considering how to create a hashable 'Blocker Key'. I think that bitwise operations on a uint64 and uint32 would be [fine](https://stackoverflow.com/questions/35586746/are-bitwise-operators-on-different-int-widths-in-c-well-defined), but decided to keep them the same for simplicity, even if this comes at the cost of some memory.

After some research, I see this is also the approach taken in the most sophisticated 9x9 open source board game implementation I readily found - Shogi program [Apery](https://github.com/HiraokaTakuya/apery_rust/blob/master/src/bitboard.rs).

Below is a Bitboard initialised to a starting position, with the repective values of __low__ and __high__ board sections printed in both base 10 and hex.

In [2]:
board = tablut.py_create_start_position()
tablut.print_board(board)

9 . . . 1 1 1 . . . 
8 . . . . 1 . . . . 
7 . . . . 1 . . . . 
6 1 . . . 1 . . . 1 
5 1 1 1 1 1 1 1 1 1 
4 1 . . . 1 . . . 1 
3 . . . . 1 . . . . 
2 . . . . 1 . . . . 
1 . . . 1 1 1 . . . 
  a b c d e f g h i
Bitboard: low: 297870862030217272 | high: 28688
Bitboard (hex): low: 0x04223FF888402038 | high: 0x0000000000007010



This should give insight into why we are using Bitboards over a 9x9 array - termed the 'mailbox' approach. We can describe every board position with two numbers. Of the reading I did to understand this method of board representation, [this article](https://analog-hors.github.io/site/magic-bitboards/) summed everything up the best, and included this cartoon:

![Jupiter](../xkcd_assigning_numbers.jpg)

To give an idea of this, let's write the date and time as a tablut board: 3092024 1502

Then we can do whatever arithmetic / bitwise operations we want on it.

In [3]:
low = 3092024
high = 1502

board = tablut.py_build_board(low, high)

low_1 = low * 42
high_1 = high - 111

board = tablut.py_build_board(low_1, high_1)

low_2 = low | low_1
high_2 = high | high_1
board = tablut.py_build_board(low_2, high_2)

9 . 1 . . . . . . . 
8 . 1 1 1 1 . 1 1 1 
7 . . . . . . . . . 
6 . . . . . . . . . 
5 . . . . . . . . . 
4 . . . . . . . . . 
3 1 1 . 1 . . . . . 
2 1 1 1 . 1 . . 1 1 
1 . . . 1 1 1 . . . 
  a b c d e f g h i
Bitboard: low: 3092024 | high: 1502
Bitboard (hex): low: 0x00000000002F2E38 | high: 0x00000000000005DE

9 . 1 . . . . . . . 
8 1 1 1 1 . 1 1 . 1 
7 . . . . . . . . . 
6 . . . . . . . . . 
5 . . . . . . . . . 
4 . . . . . . . . . 
3 1 1 1 1 . 1 1 1 1 
2 . 1 . 1 . . 1 1 . 
1 . . . . 1 1 . . 1 
  a b c d e f g h i
Bitboard: low: 129865008 | high: 1391
Bitboard (hex): low: 0x0000000007BD9530 | high: 0x000000000000056F

9 . 1 . . . . . . . 
8 1 1 1 1 1 1 1 1 1 
7 . . . . . . . . . 
6 . . . . . . . . . 
5 . . . . . . . . . 
4 . . . . . . . . . 
3 1 1 1 1 . 1 1 1 1 
2 1 1 1 1 1 . 1 1 1 
1 . . . 1 1 1 . . 1 
  a b c d e f g h i
Bitboard: low: 130006840 | high: 1535
Bitboard (hex): low: 0x0000000007BFBF38 | high: 0x00000000000005FF



## Move Generation

In chess Bitboard programs, move generation tends to be precomputed as much as possible. There are some pieces (knights, pawns, kings) which have a fixed number of candidate move squares based on piece position. For Rooks, Bishops and the Queen the situation is complicated by pieces blocking move vectors. The move generation process for these pieces centres on these 'blockers'. In Tablut, our pieces behave just like rooks, with the only excpetions being that the Throne is off limits for all non-King pieces, and it is not possible to move onto the same square as enemy pieces. Unlike chess, where the approach is to generate move sets that include the first 'blocker' piece in a file, then AND with enemy pieces for attacks, or OR with Allies for quiet moves, we should be able to perfectly describe the full set of valid moves simply from blocker configuration. However, following the chess model allows us to ignore the edge squares (as they are not blocking access to any other squares), halving our blocker permutation numbers and allowing us to use the same tables for Attacker/Defender and King moves, all at the price of one Bitwise operation.


### Blocker / Move Tables

Our next step is then to create masks for each square that describe full set of potential squares that could contain blocker pieces. We'll calculate these on the fly by iterating over indices, much the same as we would have generated moves with the mailbox approach, and then save them as constants. 

In [4]:
tablut.print_blocker_masks()

Bitboard(low = 0x00402010080402FE, high = 0x0000000000000001),
Bitboard(low = 0x00804020100804FC, high = 0x0000000000000002),
Bitboard(low = 0x01008040201008FA, high = 0x0000000000000004),
Bitboard(low = 0x02010080402010F6, high = 0x0000000000000008),
Bitboard(low = 0x04020100804020EE, high = 0x0000000000000010),
Bitboard(low = 0x08040201008040DE, high = 0x0000000000000020),
Bitboard(low = 0x10080402010080BE, high = 0x0000000000000040),
Bitboard(low = 0x201008040201007E, high = 0x0000000000000080),
Bitboard(low = 0x40201008040200FE, high = 0x0000000000000100),
Bitboard(low = 0x004020100805FC00, high = 0x0000000000000001),
Bitboard(low = 0x008040201009F800, high = 0x0000000000000002),
Bitboard(low = 0x010080402011F400, high = 0x0000000000000004),
Bitboard(low = 0x020100804021EC00, high = 0x0000000000000008),
Bitboard(low = 0x040201008041DC00, high = 0x0000000000000010),
Bitboard(low = 0x080402010081BC00, high = 0x0000000000000020),
Bitboard(low = 0x1008040201017C00, high = 0x00000000000

In [5]:
print("Square 0 Blocker Mask:")
b = tablut.py_build_board(0x00402010080402FE, 0x0000000000000001)
print("------------------------------\n\nSquare 40 Blocker Mask:")
b = tablut.py_build_board(0x04020EE080402000, 0x0000000000000010)

Square 0 Blocker Mask:
9 . . . . . . . . . 
8 1 . . . . . . . . 
7 1 . . . . . . . . 
6 1 . . . . . . . . 
5 1 . . . . . . . . 
4 1 . . . . . . . . 
3 1 . . . . . . . . 
2 1 . . . . . . . . 
1 . 1 1 1 1 1 1 1 . 
  a b c d e f g h i
Bitboard: low: 18049651735528190 | high: 1
Bitboard (hex): low: 0x00402010080402FE | high: 0x0000000000000001

------------------------------

Square 40 Blocker Mask:
9 . . . . . . . . . 
8 . . . . 1 . . . . 
7 . . . . 1 . . . . 
6 . . . . 1 . . . . 
5 . 1 1 1 . 1 1 1 . 
4 . . . . 1 . . . . 
3 . . . . 1 . . . . 
2 . . . . 1 . . . . 
1 . . . . . . . . . 
  a b c d e f g h i
Bitboard: low: 288809683492282368 | high: 16
Bitboard (hex): low: 0x04020EE080402000 | high: 0x0000000000000010



Now, let's count how many blocker bits are needed to represent the information. This is important as it informs us how large our index for each square need to be:

In [6]:
tablut.print_mask_bitcount(mask="blocker")

14, 13, 13, 13, 13, 13, 13, 13, 14, 
13, 12, 12, 12, 12, 12, 12, 12, 13, 
13, 12, 12, 12, 12, 12, 12, 12, 13, 
13, 12, 12, 12, 12, 12, 12, 12, 13, 
13, 12, 12, 12, 12, 12, 12, 12, 13, 
13, 12, 12, 12, 12, 12, 12, 12, 13, 
13, 12, 12, 12, 12, 12, 12, 12, 13, 
13, 12, 12, 12, 12, 12, 12, 12, 13, 
14, 13, 13, 13, 13, 13, 13, 13, 14, 


The idea here is to generate every permutation of blocker sets, and pre-calculate possible moves, for each square. There are a reasonable size of permutations to cache - 2^blocker_bits per square as shown above, or between 4096 and 16384 depending on the square. 

The issue comes with storing these blocker / move pairs in a compact way, as the blockers are spread out across the full uint64 range. This is where the magic comes in. If we follow the operation (blockers * magic) >> blocker_bits, we should be able to create unique indices for our moves within the range of 2^blocker_bits. 

Theoretically, it seems that we should even be able to achieve a more compact table, as there are many constructive collisions. Consider the situation where there is a blocking piece immediately next to a playing piece that is on the edge of the board. Those next 6 squares could have any combination of pieces on them, but the moves along that vector would all be the same.

Our first hurdle is that our board is 9x9 and doesn't neatly feat into a 64 bit integer. How do we use our 2 part board as a key? How do we do use both board values without losing information? This is the reason, along with decreasing table size, that I've decided to ignore the edge squares. This means that we can bitwise OR the blockers together and never have information in the same squares. Let's look at what this means for square 75:


In [7]:
tablut.py_demonstrate_blocker_merge(square=75)

Unmerged Blockers at Square 75:
9 . 1 1 . 1 1 1 1 . 
8 . . . 1 . . . . . 
7 . . . 1 . . . . . 
6 . . . 1 . . . . . 
5 . . . 1 . . . . . 
4 . . . 1 . . . . . 
3 . . . 1 . . . . . 
2 . . . 1 . . . . . 
1 . . . . . . . . . 
  a b c d e f g h i
Bitboard: low: 144397213884223488 | high: 125960
Bitboard (hex): low: 0x0201008040201000 | high: 0x000000000001EC08

---------------------
Merged Board (disregard top two rows):
9 . . . . . . . . . 
8 . . . . . . . . . 
7 . . . 1 . . . . . 
6 . . . 1 . . . . . 
5 . . . 1 . . . . . 
4 . . . 1 . . . . . 
3 . . . 1 . . . . . 
2 . 1 1 1 1 1 1 1 . 
1 . . . 1 . . . . . 
  a b c d e f g h i
Bitboard: low: 144397213884349448 | high: 0
Bitboard (hex): low: 0x020100804021FC08 | high: 0x0000000000000000



The next step is to create every permutation of blockers, which we will achieve using the Carry Rippler trick (used to find all subsets of a set) on our blocker masks, and then for each of subsets manually calculate the possible moves. Saving these in a table lets us calculate these once per square, then reference each table by index when trying to find magics.

In [8]:
square = 30
blockers, moves = tablut.py_build_bm_tables(square=square)
print(f"Blocker, Move Table Size at Square {square}: {len(blockers)}")
index = 10
print(f"Square {square} Blocker Pattern index {index}")
bb = tablut.py_build_board(blockers[10], 0)
print(f"Square {square} Moves index {index}")
bb = tablut.py_build_board(moves[10]['low'], moves[10]['high'])

Blocker, Move Table Size at Square 30: 4096
Square 30 Blocker Pattern index 10
9 . . . . . . . . . 
8 . . . . . . . . . 
7 . . . . . . . . . 
6 . . . . . . . . . 
5 . . . . . . . . . 
4 . . 1 . . . . . . 
3 . . . 1 . . . . . 
2 . . . . . . . . . 
1 . . . . . . . . . 
  a b c d e f g h i
Bitboard: low: 538968064 | high: 0
Bitboard (hex): low: 0x0000000020200000 | high: 0x0000000000000000

Square 30 Moves index 10
9 . . . 1 . . . . . 
8 . . . 1 . . . . . 
7 . . . 1 . . . . . 
6 . . . 1 . . . . . 
5 . . . 1 . . . . . 
4 . . . . 1 1 1 1 1 
3 . . . . . . . . . 
2 . . . . . . . . . 
1 . . . . . . . . . 
  a b c d e f g h i
Bitboard: low: 144397279380373504 | high: 4104
Bitboard (hex): low: 0x0201008F80000000 | high: 0x0000000000001008



### Magic Hunting

Great! Now we're ready to find out magics. We can iterate through all of the squares and try random unsigned long longs until we find one that generates unique, or constructively colliding, indexes for each blocker / move pair. We'll print these out so we can copy and paste them as constants in our Cython code.

In [9]:
magics = tablut.find_magics(action="moves")

Finding Magic for Square 80:   0%|          | 74075/100000000 [00:00<15:31, 107272.55magic/s]

MAGICS
-------------------------------------
Square 0 Magic: 0x144000420A205B00
Square 1 Magic: 0x0020008200404080
Square 2 Magic: 0x00400101020010C0
Square 3 Magic: 0x00C0020000202040
Square 4 Magic: 0x124000A100000890
Square 5 Magic: 0x088001501080D010
Square 6 Magic: 0x024000600A100068
Square 7 Magic: 0x0040000849280094
Square 8 Magic: 0x844000100202106A
Square 9 Magic: 0x5015200001140100
Square 10 Magic: 0x10001001041A8080
Square 11 Magic: 0x4042200102001040
Square 12 Magic: 0x4040A000810001A0
Square 13 Magic: 0x80212001000C0010
Square 14 Magic: 0x1400300020020018
Square 15 Magic: 0x0240200020010C18
Square 16 Magic: 0x0040201000080004
Square 17 Magic: 0x01008200080C8202
Square 18 Magic: 0x4050A0100002C500
Square 19 Magic: 0x00100208000A0040
Square 20 Magic: 0x20002A2001070102
Square 21 Magic: 0x0108001000440020
Square 22 Magic: 0x021C001001400009
Square 23 Magic: 0x0446001000200005
Square 24 Magic: 0x1040003000100018
Square 25 Magic: 0x0080002001549004
Square 26 Magic: 0x0940005000

A note here on finding magics - this initially didn't work for me. I was able to find magics for all of the squares except 17 and 53. I first attempted to use a simulated annealing algorithm to find magics, toggling bits to create neighbouring candidates. Consider the problem: trying to find specific numbers in a genuinely massive problem space. The random approach felt like throwing darts at an abyss. I left it running overnight, trialling __billions__ of candidate magics, without success or indeed getting any closer. Simulated Annealing had some interesting results. I was able to fine tune down promising candidates until I only had 256 collisions, then 72, then 28. I wasn't able to improve on this - at I don't have the mathematical grounding to analyse magics for patterns and generate magics on these grounds. As the cost for this algorithm I iterated fully through the move / blocker tables and used total collision count. To my surprise, after testing on successful magics for other squares, none of the magics used constructive collisions, leading me me to question the very way that I was capturing board information.  

On running into the problem, I made a list of candidate solutions, including: Simulated Annealing / Genetic Algorithms, improving my random generation, and increasing the hash index size. After looking at the first two I gave up and figured I would do the third, which my foolproof fallback. I bumped up the BlOCKER_BITS constants for these squares, and it still didn't work. 

Dispirited, I turned to the internet, which is where I came across [Apery](https://www.chessprogramming.org/Apery), and then the [github repo](https://github.com/HiraokaTakuya?tab=overview&from=2024-09-01&to=2024-09-02). It was fascinating to see another Bitboard implementation for a 9x9 game face the same problems, and in many cases arrive at the same solutions as I had. Notably, the 2x uint64 structure was the same, as was the approach to merge blockers from high/low boards. In fact, the approaches were so similar that I tried out some of the hardcoded rook magics in Apery, and they all worked - except for squares 17 and 53. Convinced I was going insane I then scoured Apery's code and noticed that the right shift bits had been increased by one for square 17 and 53, and in this moment realised that I had counted to squares 17 and 53 in my BLOCKER_BITS array the same way the board is printed - from the bottom left. I'd increased the indices on the wrong squares! I'm not sure why these two squares need extra indices, or indeed why none of the squares seem able to map in a way that includes constructive but not destructive collisions. But I was able to generate magics for these squares, and it was a hugely inspiring feeling to realise that my understanding has increased to the point that I can succesfully problem solve, and independently come to the same conclusions as a much more skilled practitioner.

Validating our Magics:

In [10]:
for magic in magics:
    tablut.test_magic(magic, magics[magic])

SQUARE 0: unique: 16384 | constructive: 0 | destructive: 0
SQUARE 1: unique: 8192 | constructive: 0 | destructive: 0
SQUARE 2: unique: 8192 | constructive: 0 | destructive: 0
SQUARE 3: unique: 8192 | constructive: 0 | destructive: 0
SQUARE 4: unique: 8192 | constructive: 0 | destructive: 0
SQUARE 5: unique: 8192 | constructive: 0 | destructive: 0
SQUARE 6: unique: 8192 | constructive: 0 | destructive: 0
SQUARE 7: unique: 8192 | constructive: 0 | destructive: 0
SQUARE 8: unique: 16384 | constructive: 0 | destructive: 0
SQUARE 9: unique: 8192 | constructive: 0 | destructive: 0
SQUARE 10: unique: 4096 | constructive: 0 | destructive: 0
SQUARE 11: unique: 4096 | constructive: 0 | destructive: 0
SQUARE 12: unique: 4096 | constructive: 0 | destructive: 0
SQUARE 13: unique: 4096 | constructive: 0 | destructive: 0
SQUARE 14: unique: 4096 | constructive: 0 | destructive: 0
SQUARE 15: unique: 4096 | constructive: 0 | destructive: 0
SQUARE 16: unique: 4096 | constructive: 0 | destructive: 0
SQUAR

### Building our Final Move Tables

In [2]:
MoveTables = tablut.MoveTables()

Building Move Tables...: 100%|██████████| 81/81 [00:00<00:00, 1343.17it/s]

Move Tables Initialised.


In [3]:
bb = tablut.py_create_start_position()
tablut.print_board(bb)

9 . . . 1 1 1 . . . 
8 . . . . 1 . . . . 
7 . . . . 1 . . . . 
6 1 . . . 1 . . . 1 
5 1 1 1 1 1 1 1 1 1 
4 1 . . . 1 . . . 1 
3 . . . . 1 . . . . 
2 . . . . 1 . . . . 
1 . . . 1 1 1 . . . 
  a b c d e f g h i
Bitboard: low: 297870862030217272 | high: 28688
Bitboard (hex): low: 0x04223FF888402038 | high: 0x0000000000007010



In [5]:
move_board = MoveTables.py_get_move(22, bb)
tablut.print_board(move_board)

9 . . . . . . . . . 
8 . . . . . . . . . 
7 . . . . . . . . . 
6 . . . . . . . . . 
5 . . . . . . . . . 
4 . . . . . . . . . 
3 1 1 1 1 . 1 1 1 1 
2 . . . . . . . . . 
1 . . . . . . . . . 
  a b c d e f g h i
Bitboard: low: 129761280 | high: 0
Bitboard (hex): low: 0x0000000007BC0000 | high: 0x0000000000000000



In [14]:
tablut.print_attack_masks()

cdef Bitboard[81] ATTACK_MASKS = [
Bitboard(low = 0x0000000000040004, high = 0x0000000000000000),
Bitboard(low = 0x0000000000080008, high = 0x0000000000000000),
Bitboard(low = 0x0000000000100011, high = 0x0000000000000000),
Bitboard(low = 0x0000000000200022, high = 0x0000000000000000),
Bitboard(low = 0x0000000000400044, high = 0x0000000000000000),
Bitboard(low = 0x0000000000800088, high = 0x0000000000000000),
Bitboard(low = 0x0000000001000110, high = 0x0000000000000000),
Bitboard(low = 0x0000000002000020, high = 0x0000000000000000),
Bitboard(low = 0x0000000004000040, high = 0x0000000000000000),
Bitboard(low = 0x0000000008000800, high = 0x0000000000000000),
Bitboard(low = 0x0000000010001000, high = 0x0000000000000000),
Bitboard(low = 0x0000000020002200, high = 0x0000000000000000),
Bitboard(low = 0x0000000040004400, high = 0x0000000000000000),
Bitboard(low = 0x0000000080008800, high = 0x0000000000000000),
Bitboard(low = 0x0000000100011000, high = 0x0000000000000000),
Bitboard(low = 0x000

In [15]:
tablut.print_mask_bitcount("attack")

2, 2, 3, 3, 3, 3, 3, 2, 2, 
2, 2, 3, 3, 3, 3, 3, 2, 2, 
3, 3, 4, 4, 4, 4, 4, 3, 3, 
3, 3, 4, 4, 4, 4, 4, 3, 3, 
3, 3, 4, 4, 4, 4, 4, 3, 3, 
3, 3, 4, 4, 4, 4, 4, 3, 3, 
3, 3, 4, 4, 4, 4, 4, 3, 3, 
2, 2, 3, 3, 3, 3, 3, 2, 2, 
2, 2, 3, 3, 3, 3, 3, 2, 2, 


In [16]:
allies, attacked_squares = tablut.py_build_attacked_square_tables(75)

In [17]:
for i, ally in enumerate(allies):
    tablut.py_build_board(ally, 0)
    tablut.py_build_board(attacked_squares[i]['low'], attacked_squares[i]['high'])
    print('-----------------')

9 . . . . . . . . . 
8 . . . . . . . . . 
7 . . . . . . . . . 
6 . . . . . . . . . 
5 . . . . . . . . . 
4 . . . . . . . . . 
3 . . . . . . . . . 
2 . . . . . . . . . 
1 . . . . . . . . . 
  a b c d e f g h i
Bitboard: low: 0 | high: 0
Bitboard (hex): low: 0x0000000000000000 | high: 0x0000000000000000

9 . . . . . . . . . 
8 . . . . . . . . . 
7 . . . . . . . . . 
6 . . . . . . . . . 
5 . . . . . . . . . 
4 . . . . . . . . . 
3 . . . . . . . . . 
2 . . . . . . . . . 
1 . . . . . . . . . 
  a b c d e f g h i
Bitboard: low: 0 | high: 0
Bitboard (hex): low: 0x0000000000000000 | high: 0x0000000000000000

-----------------
9 . . . . . . . . . 
8 . . . . . . . . . 
7 . . . 1 . . . . . 
6 . . . . . . . . . 
5 . . . . . . . . . 
4 . . . . . . . . . 
3 . . . . . . . . . 
2 . . . . . . . . . 
1 . . . . . . . . . 
  a b c d e f g h i
Bitboard: low: 144115188075855872 | high: 0
Bitboard (hex): low: 0x0200000000000000 | high: 0x0000000000000000

9 . . . . . . . . . 
8 . . . 1 . . . . . 
7 . . . . .

In [1]:
import tablut

In [3]:
magics = tablut.find_magics(action="attacks")

Finding Magic for Square 80:   0%|          | 109/100000000 [00:00<08:55, 186603.73magic/s]

MAGICS
-------------------------------------
Square 0 Magic: 0x2908908065008002
Square 1 Magic: 0x0CA0100480200180
Square 2 Magic: 0x4200892422000008
Square 3 Magic: 0x4222210810240000
Square 4 Magic: 0xAA38030000002000
Square 5 Magic: 0xA482011001020442
Square 6 Magic: 0x39C0002082201000
Square 7 Magic: 0x0A15004098000001
Square 8 Magic: 0x0200045079001218
Square 9 Magic: 0x0A0A021451000028
Square 10 Magic: 0x100A000489484000
Square 11 Magic: 0xC01C044200A00000
Square 12 Magic: 0x2025080080250C00
Square 13 Magic: 0x0007001280200140
Square 14 Magic: 0x12842280900000C0
Square 15 Magic: 0x0824510820000600
Square 16 Magic: 0x00C2004018880008
Square 17 Magic: 0x0000A40011100091
Square 18 Magic: 0x8340420005040000
Square 19 Magic: 0x480402000B343829
Square 20 Magic: 0x2000090100400000
Square 21 Magic: 0x0807025441040000
Square 22 Magic: 0x410024802420200A
Square 23 Magic: 0x01010208C0410018
Square 24 Magic: 0x0200246800900228
Square 25 Magic: 0x4082090020240810
Square 26 Magic: 0x0194062224

In [4]:
for magic in magics:
    tablut.test_magic(magic, magics[magic], "attacks", verbose=False)

SQUARE 0: unique: 4 | constructive: 0 | destructive: 0
SQUARE 1: unique: 4 | constructive: 0 | destructive: 0
SQUARE 2: unique: 8 | constructive: 0 | destructive: 0
SQUARE 3: unique: 8 | constructive: 0 | destructive: 0
SQUARE 4: unique: 8 | constructive: 0 | destructive: 0
SQUARE 5: unique: 8 | constructive: 0 | destructive: 0
SQUARE 6: unique: 8 | constructive: 0 | destructive: 0
SQUARE 7: unique: 4 | constructive: 0 | destructive: 0
SQUARE 8: unique: 4 | constructive: 0 | destructive: 0
SQUARE 9: unique: 4 | constructive: 0 | destructive: 0
SQUARE 10: unique: 4 | constructive: 0 | destructive: 0
SQUARE 11: unique: 8 | constructive: 0 | destructive: 0
SQUARE 12: unique: 8 | constructive: 0 | destructive: 0
SQUARE 13: unique: 8 | constructive: 0 | destructive: 0
SQUARE 14: unique: 8 | constructive: 0 | destructive: 0
SQUARE 15: unique: 8 | constructive: 0 | destructive: 0
SQUARE 16: unique: 4 | constructive: 0 | destructive: 0
SQUARE 17: unique: 4 | constructive: 0 | destructive: 0
SQ

In [1]:
import tablut

In [2]:
AttackTables = tablut.py_build_attack_tables()

Building Attack Tables...: 100%|██████████| 81/81 [00:00<00:00, 84743.98it/s]

Attack Tables Initialised.


In [3]:
bb = tablut.py_create_start_position()
tablut.print_board(bb)

9 . . . 1 1 1 . . . 
8 . . . . 1 . . . . 
7 . . . . 1 . . . . 
6 1 . . . 1 . . . 1 
5 1 1 1 1 1 1 1 1 1 
4 1 . . . 1 . . . 1 
3 . . . . 1 . . . . 
2 . . . . 1 . . . . 
1 . . . 1 1 1 . . . 
  a b c d e f g h i
Bitboard: low: 297870862030217272 | high: 28688
Bitboard (hex): low: 0x04223FF888402038 | high: 0x0000000000007010



In [4]:
attack_board = AttackTables.py_get_attacks(56, bb)
tablut.print_board(attack_board)

9 . . . . . . . . . 
8 . . . . . . . . . 
7 . . . 1 . . . . . 
6 . . 1 . . . . . . 
5 . . . . . . . . . 
4 . . . . . . . . . 
3 . . . . . . . . . 
2 . . . . . . . . . 
1 . . . . . . . . . 
  a b c d e f g h i
Bitboard: low: 144255925564211200 | high: 0
Bitboard (hex): low: 0x0200800000000000 | high: 0x0000000000000000



In [5]:
game = tablut.GameState()

In [6]:
game.print_board(unicode=True)

   a b c d e f g h i  
 9 · · · ♟ ♟ ♟ · · ·  9
 8 · · · · ♟ · · · ·  8
 7 · · · · ♙ · · · ·  7
 6 ♟ · · · ♙ · · · ♟  6
 5 ♟ ♟ ♙ ♙ ♔ ♙ ♙ ♟ ♟  5
 4 ♟ · · · ♙ · · · ♟  4
 3 · · · · ♙ · · · ·  3
 2 · · · · ♟ · · · ·  2
 1 · · · ♟ ♟ ♟ · · ·  1
   a b c d e f g h i  


In [1]:
import tablut

In [2]:
tablut.py_test_random_bitboard_lsb()

All 1000 random bitboard tests passed successfully.


In [2]:
tablut.py_main()

Building Move Tables...: 100%|██████████| 81/81 [00:00<00:00, 761.03it/s]


Move Tables Initialised.


Building Attack Tables...: 100%|██████████| 81/81 [00:00<00:00, 476491.76it/s]

Attack Tables Initialised.
   a b c d e f g h i  
 9 · · · ♟ ♟ ♟ · · ·  9
 8 · · · · ♟ · · · ·  8
 7 · · · · ♙ · · · ·  7
 6 ♟ · · · ♙ · · · ♟  6
 5 ♟ ♟ ♙ ♙ ♔ ♙ ♙ ♟ ♟  5
 4 ♟ · · · ♙ · · · ♟  4
 3 · · · · ♙ · · · ·  3
 2 · · · · ♟ · · · ·  2
 1 · · · ♟ ♟ ♟ · · ·  1
   a b c d e f g h i  
Generated 80 moves:
Move 1: d9-a9
Move 2: d9-b9
Move 3: d9-c9
Move 4: d9-d8
Move 5: d9-d7
Move 6: d9-d6
Move 7: f9-g9
Move 8: f9-h9
Move 9: f9-i9
Move 10: f9-f8
Move 11: f9-f7
Move 12: f9-f6
Move 13: e8-a8
Move 14: e8-b8
Move 15: e8-c8
Move 16: e8-d8
Move 17: e8-f8
Move 18: e8-g8
Move 19: e8-h8
Move 20: e8-i8
Move 21: a6-a9
Move 22: a6-a8
Move 23: a6-a7
Move 24: a6-b6
Move 25: a6-c6
Move 26: a6-d6
Move 27: i6-i9
Move 28: i6-i8
Move 29: i6-i7
Move 30: i6-f6
Move 31: i6-g6
Move 32: i6-h6
Move 33: b5-b9
Move 34: b5-b8
Move 35: b5-b7
Move 36: b5-b6
Move 37: b5-b4
Move 38: b5-b3
Move 39: b5-b2
Move 40: b5-b1
Move 41: h5-h9
Move 42: h5-h8
Move 43: h5-h7
Move 44: h5-h6
Move 45: h5-h4
Move 46: h5-h3
Move 47: h

In [12]:
tablut.py_build_board(0x7FFFFEFFFFFFFFFF, 0x000000000003FFFF)

9 1 1 1 1 1 1 1 1 1 
8 1 1 1 1 1 1 1 1 1 
7 1 1 1 1 1 1 1 1 1 
6 1 1 1 1 1 1 1 1 1 
5 1 1 1 1 . 1 1 1 1 
4 1 1 1 1 1 1 1 1 1 
3 1 1 1 1 1 1 1 1 1 
2 1 1 1 1 1 1 1 1 1 
1 1 1 1 1 1 1 1 1 1 
  a b c d e f g h i
Bitboard: low: 9223370937343148031 | high: 262143
Bitboard (hex): low: 0x7FFFFEFFFFFFFFFF | high: 0x000000000003FFFF



{'low': 9223370937343148031, 'high': 262143}

In [10]:
tablut.py_test_pop_lsb()

9 . . . . . . . . . 
8 . . . . . . . . . 
7 . . . . . . . . . 
6 . . . . . . . . . 
5 . . . . . . . . . 
4 . . . . . . . . . 
3 . . . . . . . . . 
2 . . . . . . . . . 
1 . . . . . . . . . 
  a b c d e f g h i
Bitboard: low: 0 | high: 0
Bitboard (hex): low: 0x0000000000000000 | high: 0x0000000000000000

-------------------
9 1 1 1 1 1 1 1 1 1 
8 1 1 1 1 1 1 1 1 1 
7 1 1 1 1 1 1 1 1 1 
6 1 1 1 1 1 1 1 1 1 
5 1 1 1 1 1 1 1 1 1 
4 1 1 1 1 1 1 1 1 1 
3 1 1 1 1 1 1 1 1 1 
2 1 1 1 1 1 1 1 1 1 
1 1 1 1 1 1 1 1 1 1 
  a b c d e f g h i
Bitboard: low: 9223372036854775807 | high: 262143
Bitboard (hex): low: 0x7FFFFFFFFFFFFFFF | high: 0x000000000003FFFF

-------------------
9 1 1 1 1 1 1 1 1 1 
8 1 1 1 1 1 1 1 1 1 
7 1 1 1 1 1 1 1 1 1 
6 1 1 1 1 1 1 1 1 1 
5 1 1 1 1 1 1 1 1 1 
4 1 1 1 1 1 1 1 1 1 
3 1 1 1 1 1 1 1 1 1 
2 1 1 1 1 1 1 1 1 1 
1 . 1 1 1 1 1 1 1 1 
  a b c d e f g h i
Bitboard: low: 9223372036854775806 | high: 262143
Bitboard (hex): low: 0x7FFFFFFFFFFFFFFE | high: 0x000000000003FFFF

0


In [7]:
chr((22%9) + ord('a'))

'e'

In [4]:
def bitstring_recursion(remainder, bitstring):
    for i in range(remainder):
        if 2**i == remainder:
            bitstring[i] = '1'
            return bitstring
        elif 2**i > remainder:
            bitstring[i-1] = '1'
            remainder = remainder - 2**(i-1)
            return bitstring_recursion(remainder, bitstring)

def int_to_bitstring(x):
    for i in range(x):
        if 2 ** i == x:
            bitstring = ['0' for i in range(i+1)]
            bitstring[i] = '1'
            break
        if 2 ** i > x:
            bitstring = ['0' for i in range(i)]
            bitstring[i-1] = '1'
            remainder = x - 2**(i-1)
            bitstring_recursion(remainder, bitstring)
            break
    return ''.join(bitstring)

In [17]:
len(int_to_bitstring(16384))

15

In [18]:
len(int_to_bitstring(79) + int_to_bitstring(80))

14

In [7]:
(80 << 7) | 79

10319

In [19]:
hex(1 << 14)

'0x4000'

# cython: language_level=3
from libc.stdlib cimport malloc, free
from libc.stdint cimport uint64_t
from cython.parallel import prange

cdef struct Bitboard:
    uint64_t low
    uint64_t high

cdef struct MoveTable:
    Bitboard* moves
    int size

cdef struct MoveTables:
    MoveTable[81] tables

cdef struct KeyBoardTable:
    uint64_t* key
    Bitboard* board
    int size

# Assume these are defined elsewhere
cdef uint64_t[81] MAGIC
cdef int[81] SHIFT_BITS
cdef Bitboard[81] BLOCKER_MASK
cdef int[81] BLOCKER_BITS

cdef MoveTables* create_move_tables() nogil:
    cdef MoveTables* tables = <MoveTables*>malloc(sizeof(MoveTables))
    if tables == NULL:
        return NULL

    cdef int i
    for i in range(81):
        tables.tables[i].moves = NULL
        tables.tables[i].size = 0

    return tables

cdef void free_move_tables(MoveTables* tables) nogil:
    if tables != NULL:
        cdef int i
        for i in range(81):
            if tables.tables[i].moves != NULL:
                free(tables.tables[i].moves)
        free(tables)

cdef void set_table(MoveTables* tables, int square, KeyBoardTable* blocker_moves, int size) nogil:
    if tables.tables[square].moves != NULL:
        free(tables.tables[square].moves)

    tables.tables[square].moves = <Bitboard*>malloc(size * sizeof(Bitboard))
    if tables.tables[square].moves == NULL:
        # Handle memory allocation failure
        return

    tables.tables[square].size = size

    cdef int i, index
    for i in range(blocker_moves.size):
        index = get_hash_value(blocker_moves.key[i], MAGIC[square], SHIFT_BITS[square])
        tables.tables[square].moves[index] = blocker_moves.board[i]

cdef Bitboard get_move(MoveTables* tables, int square, Bitboard occupancy) nogil:
    cdef uint64_t blocker_key = merge(get_blockers(occupancy, square))
    cdef int index = get_hash_value(blocker_key, MAGIC[square], SHIFT_BITS[square])
    return tables.tables[square].moves[index]

cdef void build_move_tables(MoveTables* tables) nogil:
    cdef KeyBoardTable blocker_moves
    cdef int i, size
    for i in prange(81, nogil=True):
        blocker_moves = build_key_board_table(i, BLOCKER_MASK[i], BLOCKER_BITS[i], find_moves_otf)
        size = 1 << (64 - SHIFT_BITS[i])
        set_table(tables, i, &blocker_moves, size)
        free_key_board_table(&blocker_moves)

# Assume these functions are defined elsewhere
cdef uint64_t get_hash_value(uint64_t key, uint64_t magic, int shift) nogil:
    # Implementation
    pass

cdef Bitboard get_blockers(Bitboard occupancy, int square) nogil:
    # Implementation
    pass

cdef uint64_t merge(Bitboard b) nogil:
    # Implementation
    pass

cdef KeyBoardTable build_key_board_table(int square, Bitboard mask, int bits, 
                                         Bitboard (*find_moves)(int, Bitboard)) nogil:
    # Implementation
    pass

cdef void free_key_board_table(KeyBoardTable* table) nogil:
    # Implementation
    pass

# Python-accessible functions
def py_create_and_build_move_tables():
    cdef MoveTables* tables = create_move_tables()
    if tables == NULL:
        raise MemoryError("Failed to allocate MoveTables")
    
    build_move_tables(tables)
    return <size_t>tables  # Return as integer for Python

def py_free_move_tables(size_t ptr):
    free_move_tables(<MoveTables*>ptr)

def py_get_move(size_t ptr, int square, uint64_t occupancy_low, uint64_t occupancy_high):
    cdef MoveTables* tables = <MoveTables*>ptr
    cdef Bitboard occupancy
    occupancy.low = occupancy_low
    occupancy.high = occupancy_high
    cdef Bitboard result = get_move(tables, square, occupancy)
    return result.low, result.high

## Materials

Cython documentation
https://cython.readthedocs.io/en/latest/index.html

Magic BB Wiki


Bit Twiddling
https://graphics.stanford.edu/~seander/bithacks.html

Apery github

Youtube series

Simulated Annealing Article